# 03 · ResNet50V2 (transfer learning 3/5)

Misma arquitectura de cabeza y mismo régimen de entrenamiento que `01_mobilenet.ipynb`, cambiando solo el backbone. ResNet50V2 es el baseline clásico de la familia residual, más grande que MobileNet.

In [1]:
import os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)

print("Raíz del proyecto:", ROOT)

Raíz del proyecto: /Users/cristiansandoval/Universidad/Semillero/ButterflyModeling


In [2]:
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import tensorflow as tf

from scripts import corpus, evalStats
from scripts import modelRegistry as registro
from scripts.architectures import buildModel, preprocessFn
from scripts.dataPipeline import buildAugmenter, loadSplit, preparar
from scripts.vizStyle import SERIE_1, SERIE_2, applyStyle, plotConfusionMatrix

ARQUITECTURA = "resnet50_v2"
RUN_NAME = "resnet50-v2"
NOTAS = "Transfer learning: ResNet50V2 congelado + la misma cabeza del baseline"
PROMOVER = False

TAMANO = (320, 320)
LOTE = 32
EPOCAS = 50
PACIENCIA = 8
TASA_APRENDIZAJE = 1e-3
SEMILLA = 42

tf.keras.utils.set_random_seed(SEMILLA)
corpus.materializar()

RUN_ID = registro.buildRunId(RUN_NAME)
RUN_DIR = registro.createRunDir(RUN_ID)
IMGS = RUN_DIR / "imgs"

print(f"run_id : {RUN_ID}")
print(f"TF     : {tf.__version__} · dispositivos: "
      f"{[d.device_type for d in tf.config.list_physical_devices()]}")

run_id : 20260902T110140Z_resnet50-v2
TF     : 2.19.1 · dispositivos: ['CPU']


## 1 · Datos

Un tf.data.Dataset por split. Solo `train` se aumenta; `validate`/`test` son
deterministas (`shuffle=False`, indispensable para que `y_true` se alinee con
las predicciones).

In [3]:
preprocess = preprocessFn(ARQUITECTURA)
augmentador = buildAugmenter()

dsTrainCrudo, clases = loadSplit("dataset/train", TAMANO, LOTE)
dsValCrudo, _ = loadSplit("dataset/validate", TAMANO, LOTE, shuffle=False)
dsTestCrudo, _ = loadSplit("dataset/test", TAMANO, LOTE, shuffle=False)

dsTrain = preparar(dsTrainCrudo, preprocess, augmentador)
dsVal = preparar(dsValCrudo, preprocess)
dsTest = preparar(dsTestCrudo, preprocess)

print(f"{len(clases)} clases")

Found 7218 files belonging to 79 classes.
Found 1038 files belonging to 79 classes.
Found 2062 files belonging to 79 classes.
79 clases


## 2 · Modelo

In [4]:
modelo = buildModel(ARQUITECTURA, len(clases))

entrenables = sum(int(np.prod(w.shape)) for w in modelo.trainable_weights)
congelados = sum(int(np.prod(w.shape)) for w in modelo.non_trainable_weights)
pd.Series({
    "parametros_totales": modelo.count_params(),
    "entrenables": entrenables,
    "congelados": congelados,
    "clases_salida": len(clases),
}).to_frame("valor")

,valor
parametros_totales,24791887
entrenables,1225295
congelados,23566592
clases_salida,79


## 3 · Pesos de clase

In [5]:
pesosClase = corpus.classWeights()
pd.Series(
    {clases[i]: round(p, 3) for i, p in pesosClase.items()}
).sort_values(ascending=False).head(8).to_frame("peso")

,peso
autochton_itylus,3.151
taygetis_sylvia,3.151
cissia_confusa,3.151
pseudodebis_puritana,3.046
pyrgus_adepta,2.947
pteronymia_latilla,2.125
euselasia_mys,2.077
pteronymia_aletta,1.986


## 4 · Entrenamiento

In [6]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

modelo.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=TASA_APRENDIZAJE),
    metrics=["accuracy"],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=PACIENCIA,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3,
                       min_lr=1e-6, verbose=1),
]

inicio = datetime.now(timezone.utc)
ajuste = modelo.fit(
    dsTrain,
    validation_data=dsVal,
    epochs=EPOCAS,
    callbacks=callbacks,
    class_weight=pesosClase,
)
fin = datetime.now(timezone.utc)

historia = {k: [float(v) for v in vs] for k, vs in ajuste.history.items()}
print(f"\n{len(historia['loss'])} épocas en {(fin - inicio).total_seconds() / 60:.1f} min")

Epoch 1/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 475s 2s/step - accuracy: 0.1277 - loss: 3.9065 - val_accuracy: 0.4355 - val_loss: 2.2006 - learning_rate: 0.0010
Epoch 2/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 587s 3s/step - accuracy: 0.3071 - loss: 2.6776 - val_accuracy: 0.5366 - val_loss: 1.6758 - learning_rate: 0.0010
Epoch 3/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 636s 3s/step - accuracy: 0.4094 - loss: 2.2168 - val_accuracy: 0.5809 - val_loss: 1.4652 - learning_rate: 0.0010
Epoch 4/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 671s 3s/step - accuracy: 0.4622 - loss: 1.9370 - val_accuracy: 0.6329 - val_loss: 1.2691 - learning_rate: 0.0010
Epoch 5/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 668s 3s/step - accuracy: 0.4994 - loss: 1.7887 - val_accuracy: 0.6590 - val_loss: 1.1628 - learning_rate: 0.0010
Epoch 6/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 660s 3s/step - accuracy: 0.5209 - loss: 1.6628 - val_accuracy: 0.6676 - val_loss: 1.1007 - learning_rate: 0.0010
Epoch 7/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 669s 3s/step - accuracy: 0.5413 - loss: 1.

KeyboardInterrupt: 

## 5 · Curvas de entrenamiento

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

epocas = range(1, len(historia["loss"]) + 1)
figura, ejes = plt.subplots(1, 2, figsize=(11, 4.2))

ejes[0].plot(epocas, historia["accuracy"], color=SERIE_1, label="Entrenamiento")
ejes[0].plot(epocas, historia["val_accuracy"], color=SERIE_2, label="Validación")
ejes[0].set(title=f"{RUN_NAME} · Exactitud", xlabel="Época", ylim=(0, 1))
ejes[0].legend(loc="lower right")

ejes[1].plot(epocas, historia["loss"], color=SERIE_1, label="Entrenamiento")
ejes[1].plot(epocas, historia["val_loss"], color=SERIE_2, label="Validación")
ejes[1].set(title=f"{RUN_NAME} · Pérdida", xlabel="Época")
ejes[1].legend(loc="upper right")

figura.tight_layout()
figura.savefig(IMGS / "training_history.png", bbox_inches="tight")
plt.show()

## 6 · Evaluación

Dos conjuntos: `test` (20%, métrica principal) y `validate` (10%, el que
monitorean los callbacks).

In [ ]:
metricas = {
    "test": evalStats.evaluar(modelo, dsTest, clases),
    "validate": evalStats.evaluar(modelo, dsVal, clases),
}

pd.DataFrame([
    {k: round(v, 4) if isinstance(v, float) else v
     for k, v in m.items()
     if k in ("num_samples", "accuracy", "top3_accuracy",
              "macro_f1", "weighted_f1", "macro_precision", "macro_recall")}
    for m in metricas.values()
], index=list(metricas.keys()))

## 7 · Matriz de confusión

In [ ]:
_ = plotConfusionMatrix(
    metricas["test"],
    f"Matriz de confusión · prueba · {RUN_NAME}",
)
plt.savefig(IMGS / "confusion_test.png", bbox_inches="tight")
plt.show()

## 8 · Confusiones principales

In [ ]:
confusiones = evalStats.confusionesPrincipales(metricas["test"])
pd.DataFrame(confusiones).head(10)

## 9 · Versionado

In [ ]:
modelo.save(RUN_DIR / "model.keras")
registro.writeJson(RUN_DIR / "history.json", historia)
registro.writeJson(RUN_DIR / "metrics.json", {
    "run_id": RUN_ID,
    "class_names": clases,
    "splits": metricas,
    "top_confusions": {
        s: evalStats.confusionesPrincipales(m) for s, m in metricas.items()
    },
})

configuracion = {
    "run_id": RUN_ID,
    "name": RUN_NAME,
    "architecture": ARQUITECTURA,
    "notes": NOTAS,
    "num_classes": len(clases),
    "hyperparameters": {
        "img_size": list(TAMANO),
        "batch_size": LOTE,
        "epochs_budget": EPOCAS,
        "epochs_ran": len(historia["loss"]),
        "learning_rate": TASA_APRENDIZAJE,
        "early_stopping_patience": PACIENCIA,
        "class_weights": "balanced",
        "seed": SEMILLA,
        "backbone": f"{ARQUITECTURA}/imagenet (congelado)",
    },
    "environment": {
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "platform": platform.platform(),
    },
    "started_at": inicio.isoformat(),
    "finished_at": fin.isoformat(),
    "training_minutes": round((fin - inicio).total_seconds() / 60, 1),
}
registro.writeJson(RUN_DIR / "run.json", configuracion)

registro.registerRun({
    "run_id": RUN_ID,
    "name": RUN_NAME,
    "architecture": ARQUITECTURA,
    "num_classes": len(clases),
    "epochs_ran": len(historia["loss"]),
    "training_minutes": configuracion["training_minutes"],
    "test_accuracy": metricas["test"]["accuracy"],
    "test_macro_f1": metricas["test"]["macro_f1"],
    "validate_accuracy": metricas["validate"]["accuracy"],
    "validate_macro_f1": metricas["validate"]["macro_f1"],
    "notes": NOTAS,
    "created_at": inicio.isoformat(),
})

if PROMOVER:
    registro.promoteRun(RUN_ID)

print(f"Run guardado en {RUN_DIR}")

## Resumen

In [ ]:
pd.Series({
    "run_id": RUN_ID,
    "arquitectura": ARQUITECTURA,
    "clases": len(clases),
    "epocas": len(historia["loss"]),
    "minutos": configuracion["training_minutes"],
    "test_accuracy": round(metricas["test"]["accuracy"], 4),
    "test_top3": round(metricas["test"]["top3_accuracy"], 4),
    "test_macro_f1": round(metricas["test"]["macro_f1"], 4),
    "validate_accuracy": round(metricas["validate"]["accuracy"], 4),
    "validate_macro_f1": round(metricas["validate"]["macro_f1"], 4),
}).to_frame("valor")